# 09 · EORI check

Research log: [Step 10](../docs/research_log.md#step-10--eori) (in Romanian). Data and checks as of October 2026.

**Question.** Can EORI numbers (customs IDs, assumed GB + VAT + 000) give the VAT number of companies that do not publish it?

**Method.** Same 3 WET files as Step 2; regex for "EORI" + GB/XI + 12 digits; suffix and checksum of the 9-digit base checked.

**Result (from the research log).**
- 8 pages mention EORI, against ~676 that mention VAT.
- 1 EORI found (thedjshop.co.uk): GB + the VAT number + 000, the same VAT already found in Step 2.
- EORI brings no new numbers: dead end for coverage.

**Notes for re-running.**
- The first cell is the first version (substring `"eori"`, which also matches "theories": 641 pages); the second cell is the corrected version.
- Reads the WET files in `../data/wet/` (see notebook 05).

*Cells are kept as they were run, outputs included; only file paths were updated to the new `data/` layout and session timestamps in outputs were cut to the month.*

In [15]:
import re, glob
import pandas as pd
from pathlib import Path
from warcio.archiveiterator import ArchiveIterator

def checksum_ok(nr9):
    if len(nr9) != 9 or not nr9.isdigit():
        return False
    s = sum(int(d) * w for d, w in zip(nr9[:7], range(8, 1, -1))) + int(nr9[7:])
    return s % 97 == 0 or (s + 55) % 97 == 0

RE_EORI = re.compile(r"\bEORI\b[^A-Z0-9]{0,40}?(GB|XI)[\s\-]?((?:\d[\s\-]?){11}\d)(?![\s\-]?\d)", re.I)
RE_VAT_CUV = re.compile(r"\bVAT\b", re.I)

eori, pagini_eori_cuvant, pagini = [], 0, 0
for fisier in sorted(glob.glob("../data/wet/*.warc.wet.gz")):
    with open(fisier, "rb") as f:
        for rec in ArchiveIterator(f):
            if rec.rec_type != "conversion":
                continue
            pagini += 1
            text = rec.content_stream().read().decode("utf-8", errors="replace")
            if "eori" not in text.lower():
                continue
            pagini_eori_cuvant += 1
            url = rec.rec_headers.get_header("WARC-Target-URI")
            for m in RE_EORI.finditer(text):
                cifre = re.sub(r"\D", "", m.group(2))
                baza, sufix = cifre[:9], cifre[9:]
                ctx = text[max(0, m.start() - 200): m.end() + 200].replace("\n", " ")
                eori.append({"fisier": Path(fisier).name, "url": url, "prefix": m.group(1).upper(),
                             "eori": cifre, "baza9": baza, "sufix3": sufix,
                             "sufix_000": sufix == "000", "checksum_baza": checksum_ok(baza),
                             "vat_pe_pagina": bool(RE_VAT_CUV.search(text)), "context": ctx})

df_e = pd.DataFrame(eori)
print("Pagini parcurse:", pagini, "| pagini cu cuvântul EORI:", pagini_eori_cuvant)
print("EORI găsite:", len(df_e), "| unice:", df_e.eori.nunique() if len(df_e) else 0)
if len(df_e):
    print("Sufix 000:", df_e.sufix_000.sum(), "| baza trece checksum TVA:", df_e.checksum_baza.sum())
    tva_pas2 = set(pd.read_csv("../data/results/cc_pas2_candidati.csv", dtype=str).vat)
    df_e["baza_e_in_tva_pas2"] = df_e.baza9.isin(tva_pas2)
    print("Baza EORI e printre TVA-urile din Pasul 2:", df_e.baza_e_in_tva_pas2.sum())
    pd.set_option("display.max_colwidth", 120)
    print(df_e[["url", "prefix", "eori", "sufix_000", "checksum_baza", "vat_pe_pagina"]].to_string(index=False))
    df_e.to_csv("../data/results/eori_pas2_wet.csv", index=False)

Pagini parcurse: 64082 | pagini cu cuvântul EORI: 641
EORI găsite: 0 | unice: 0


In [ ]:
import re, glob
import pandas as pd
from pathlib import Path
from warcio.archiveiterator import ArchiveIterator

def checksum_ok(nr9):
    if len(nr9) != 9 or not nr9.isdigit():
        return False
    s = sum(int(d) * w for d, w in zip(nr9[:7], range(8, 1, -1))) + int(nr9[7:])
    return s % 97 == 0 or (s + 55) % 97 == 0

RE_EORI_CUV = re.compile(r"\bEORI\b", re.I)
RE_EORI = re.compile(r"\bEORI\b[^0-9]{0,40}?(GB|XI)[\s\-]?((?:\d[\s\-]?){11}\d)(?![\s\-]?\d)", re.I)
RE_VAT_CUV = re.compile(r"\bVAT\b", re.I)

eori, pagini_eori_cuvant, pagini = [], 0, 0
for fisier in sorted(glob.glob("../data/wet/*.warc.wet.gz")):
    with open(fisier, "rb") as f:
        for rec in ArchiveIterator(f):
            if rec.rec_type != "conversion":
                continue
            pagini += 1
            text = rec.content_stream().read().decode("utf-8", errors="replace")
            if not RE_EORI_CUV.search(text):
                continue
            pagini_eori_cuvant += 1
            url = rec.rec_headers.get_header("WARC-Target-URI")
            for m in RE_EORI.finditer(text):
                cifre = re.sub(r"\D", "", m.group(2))
                baza, sufix = cifre[:9], cifre[9:]
                ctx = text[max(0, m.start() - 200): m.end() + 200].replace("\n", " ")
                eori.append({"fisier": Path(fisier).name, "url": url, "prefix": m.group(1).upper(),
                             "eori": cifre, "baza9": baza, "sufix3": sufix,
                             "sufix_000": sufix == "000", "checksum_baza": checksum_ok(baza),
                             "vat_pe_pagina": bool(RE_VAT_CUV.search(text)), "context": ctx})

df_e = pd.DataFrame(eori)
print("Pagini parcurse:", pagini, "| pagini cu cuvântul EORI:", pagini_eori_cuvant)
print("EORI găsite:", len(df_e), "| unice:", df_e.eori.nunique() if len(df_e) else 0)
if len(df_e):
    print("Sufix 000:", df_e.sufix_000.sum(), "| baza trece checksum TVA:", df_e.checksum_baza.sum())
    tva_pas2 = set(pd.read_csv("../data/results/cc_pas2_candidati.csv", dtype=str).vat)
    df_e["baza_e_in_tva_pas2"] = df_e.baza9.isin(tva_pas2)
    print("Baza EORI e printre TVA-urile din Pasul 2:", df_e.baza_e_in_tva_pas2.sum())
    pd.set_option("display.max_colwidth", 120)
    print(df_e[["url", "prefix", "eori", "sufix_000", "checksum_baza", "vat_pe_pagina"]].to_string(index=False))
    df_e.to_csv("../data/results/eori_pas2_wet.csv", index=False)

Pagini parcurse: 64082 | pagini cu cuvântul EORI: 8
EORI găsite: 1 | unice: 1
Sufix 000: 1 | baza trece checksum TVA: 1
Baza EORI e printre TVA-urile din Pasul 2: 1
                                               url prefix         eori  sufix_000  checksum_baza  vat_pe_pagina
https://www.thedjshop.co.uk/novation-flkey-49.html     GB 717899083000       True           True           True
